# Stage 0 — Data Audit & Schema Recovery (PRAGMA-inspired)

**Goal.** Before any PRAGMA-style work (key–value–time tokenisation, typed event
encoders, customer event histories, masked-event pretraining), audit what the
current processed transaction dataset actually contains: columns, Arrow types,
row counts, null rates, approximate cardinalities, timestamp/amount/identifier
coverage, and duplicate / near-duplicate transaction descriptions.

**PRAGMA context** (see `docs/PRAGMA-Revolut Foundation Model.pdf`): PRAGMA
represents each customer as an *event history* plus *profile state*, where every
event is a timestamp plus key–value pairs (`Type`, `Direction`, `Amount`,
`Currency`, `Fee`, `Description`, `MCC`, …). That design imposes concrete data
requirements which this audit checks field by field.

**Guarantees.**
- The source parquet is opened **read-only** and streamed with PyArrow batches —
  the full dataset is never loaded into pandas.
- All outputs are written only under `ml_model/data/processed/audit/`.
- Existing labelled datasets and model notebooks are not modified.
- Deterministic: fixed seeds and a fixed hash key for all sampling/sketching.

**Schema-recovery note.** The canonical training input referenced by the other
notebooks is `transactions_for_embeddings.parquet` (output of
`00_apply_regex_rules.ipynb`). If it is absent on this machine the audit falls
back to its upstream superset `labelled_transactions_v2_regex_corrected.parquet`
and records which file was actually audited in the schema report.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# ---------------------------------------------------------------- determinism
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
HASH_KEY = "galahad-audit-42"  # 16 chars, fixed key for pandas 64-bit hashing

# ---------------------------------------------------------------- parameters
BATCH_SIZE = 1_048_576          # rows per streamed Arrow batch (~1 row group)
KMV_K = 8192                    # k-minimum-values sketch size (distinct counts)
TOP_CAP, TOP_PRUNE_TO = 60_000, 30_000   # bounded top-value counters
AMOUNT_SAMPLE_STRIDE = 50       # deterministic stride sample for quantiles
AMOUNT_SAMPLE_CAP = 1_200_000
TEMPLATE_EXAMPLE_CAP = 50_000
N_EXAMPLES = 8
# Optional smoke-test override: process only the first N batches.
MAX_BATCHES = int(os.environ.get("GALAHAD_AUDIT_MAX_BATCHES", "0")) or None

# ---------------------------------------------------------------- paths
def find_ml_model_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if cand.name == "ml_model" and (cand / "data" / "processed").is_dir():
            return cand
        if (cand / "ml_model" / "data" / "processed").is_dir():
            return cand / "ml_model"
    raise FileNotFoundError(f"ml_model root not found above {start}")

ML_MODEL_ROOT = find_ml_model_root(Path.cwd().resolve())
PROCESSED_DIR = (ML_MODEL_ROOT / "data" / "processed").resolve()
AUDIT_DIR = PROCESSED_DIR / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
assert AUDIT_DIR.resolve().is_relative_to(PROCESSED_DIR), "audit dir escaped processed/"

DATASET_CANDIDATES = [
    PROCESSED_DIR / "transactions_for_embeddings.parquet",
    PROCESSED_DIR / "labelled_transactions_v2_regex_corrected.parquet",
]
DATASET_PATH = next((p for p in DATASET_CANDIDATES if p.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(f"No dataset found among {DATASET_CANDIDATES}")
assert os.access(DATASET_PATH, os.R_OK), f"unreadable: {DATASET_PATH}"

# ---------------------------------------------------------------- logging
log = logging.getLogger("stage0_audit")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
for h in (logging.StreamHandler(), logging.FileHandler(AUDIT_DIR / "audit_run.log", mode="w")):
    h.setFormatter(_fmt)
    log.addHandler(h)

log.info("dataset: %s", DATASET_PATH)
log.info("requested primary (%s) present: %s",
         DATASET_CANDIDATES[0].name, DATASET_CANDIDATES[0].is_file())
log.info("audit outputs: %s", AUDIT_DIR)
if MAX_BATCHES:
    log.warning("SMOKE-TEST MODE: only %d batches will be processed", MAX_BATCHES)

## 1. File-level metadata and Arrow schema

Read parquet footer metadata only (no row data): row count, row groups,
compression, and the full Arrow schema.

In [ ]:
pf = pq.ParquetFile(DATASET_PATH)
meta = pf.metadata
schema = pf.schema_arrow

row_group_info = []
for rg_idx in range(meta.num_row_groups):
    rg = meta.row_group(rg_idx)
    row_group_info.append({
        "row_group": rg_idx,
        "num_rows": rg.num_rows,
        "total_byte_size": rg.total_byte_size,
    })

col_meta = []
rg0 = meta.row_group(0)
for i, name in enumerate(schema.names):
    cc = rg0.column(i)
    col_meta.append({
        "column": name,
        "arrow_type": str(schema.types[i]),
        "parquet_physical_type": cc.physical_type,
        "compression": cc.compression,
    })

schema_report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "dataset_path": str(DATASET_PATH),
    "dataset_is_requested_primary": DATASET_PATH == DATASET_CANDIDATES[0],
    "requested_primary": str(DATASET_CANDIDATES[0]),
    "file_size_bytes": DATASET_PATH.stat().st_size,
    "file_mtime": datetime.fromtimestamp(DATASET_PATH.stat().st_mtime, tz=timezone.utc).isoformat(),
    "num_rows": meta.num_rows,
    "num_row_groups": meta.num_row_groups,
    "created_by": meta.created_by,
    "columns": col_meta,
    "row_groups": row_group_info,
}

log.info("rows=%s row_groups=%s size=%.1f MB columns=%d",
         f"{meta.num_rows:,}", meta.num_row_groups,
         DATASET_PATH.stat().st_size / 1e6, len(schema.names))
pd.DataFrame(col_meta)

## 2. Streaming profilers

Memory-safe helpers used by the single streaming pass:

- **`KMVSketch`** — k-minimum-values estimator for approximate distinct counts
  (exact while cardinality < *k*), fed by deterministic 64-bit hashes.
- **`BoundedCounter`** — heavy-hitter counter with a hard cap; pruned counts are
  flagged approximate.
- **`NumericProfile`** — streaming min/max/mean/std/zero/negative accumulators.
- **`normalise_desc`** — collapses digits and punctuation so near-duplicate
  descriptions (e.g. the same merchant with different reference numbers) map to
  one template.

In [ ]:
class KMVSketch:
    """Approximate distinct count via the k smallest 64-bit hashes."""

    def __init__(self, k: int = KMV_K):
        self.k = k
        self.hashes = np.empty(0, dtype=np.uint64)

    def update(self, hashes: np.ndarray) -> None:
        merged = np.unique(np.concatenate([self.hashes, hashes.astype(np.uint64)]))
        self.hashes = merged[: self.k]

    @property
    def is_exact(self) -> bool:
        return len(self.hashes) < self.k

    def estimate(self) -> int:
        if self.is_exact:
            return int(len(self.hashes))
        kth = float(self.hashes[-1])
        return int((self.k - 1) / (kth / 2**64))


class BoundedCounter:
    """Counter capped in memory; prunes to heavy hitters when over cap."""

    def __init__(self, cap: int = TOP_CAP, prune_to: int = TOP_PRUNE_TO):
        self.cap, self.prune_to = cap, prune_to
        self.counter: Counter = Counter()
        self.truncated = False

    def update_pairs(self, values, counts) -> None:
        self.counter.update(dict(zip(values, counts)))
        if len(self.counter) > self.cap:
            self.counter = Counter(dict(self.counter.most_common(self.prune_to)))
            self.truncated = True


class NumericProfile:
    def __init__(self):
        self.n = 0
        self.sum = 0.0
        self.sumsq = 0.0
        self.min = np.inf
        self.max = -np.inf
        self.zeros = 0
        self.negatives = 0

    def update(self, x: np.ndarray) -> None:
        if len(x) == 0:
            return
        self.n += len(x)
        self.sum += float(x.sum())
        self.sumsq += float((x.astype(np.float64) ** 2).sum())
        self.min = min(self.min, float(x.min()))
        self.max = max(self.max, float(x.max()))
        self.zeros += int((x == 0).sum())
        self.negatives += int((x < 0).sum())

    def summary(self) -> dict:
        if self.n == 0:
            return {"count": 0}
        mean = self.sum / self.n
        var = max(self.sumsq / self.n - mean**2, 0.0)
        return {
            "count": self.n, "mean": mean, "std": var**0.5,
            "min": self.min, "max": self.max,
            "zeros": self.zeros, "negatives": self.negatives,
        }


def hash_series(s: pd.Series) -> np.ndarray:
    """Deterministic 64-bit hashes (fixed key => reproducible across runs)."""
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype("string").fillna("")
    return pd.util.hash_pandas_object(s, index=False, hash_key=HASH_KEY).to_numpy(np.uint64)


_norm_digits = re.compile(r"\d+")
_norm_junk = re.compile(r"[^a-z#]+")


def normalise_desc(s: pd.Series) -> pd.Series:
    """Lowercase, digits -> '#', punctuation -> space, collapse whitespace."""
    out = s.astype("string").str.lower()
    out = out.str.replace(_norm_digits, "#", regex=True)
    out = out.str.replace(_norm_junk, " ", regex=True)
    return out.str.strip().str.replace(r"\s+", " ", regex=True)


def detect_date_format(sample: pd.Series) -> str | None:
    """Pick the strptime format that parses the largest share of a sample."""
    candidates = ["%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y", "%Y-%m-%d %H:%M:%S", "%Y%m%d"]
    best, best_rate = None, 0.0
    for fmt in candidates:
        parsed = pd.to_datetime(sample, format=fmt, errors="coerce")
        rate = parsed.notna().mean()
        if rate > best_rate:
            best, best_rate = fmt, rate
    return best if best_rate >= 0.9 else None

print("profilers ready")

## 3. Single streaming pass over all row groups

One pass computes everything at once, per column: null/empty counts, KMV
distinct estimates, bounded top-value counters and example values — plus
dedicated accumulators for the timestamp column, the amount column
(including an amount-sign × debit/credit cross-tab and a deterministic stride
sample for quantiles), and raw/normalised description sketches for the
duplicate analysis. Peak memory stays bounded regardless of dataset size.

In [ ]:
COL_NAMES = list(schema.names)
DESC_COL = "TransactionParticulars"
DATE_COL = "TransactionDate"
AMOUNT_COL = "TransactionAmount"
DIRECTION_COL = "TransactionTypeDebitCredit"

profiles = {
    name: {
        "rows": 0, "nulls": 0, "empties": 0,
        "kmv": KMVSketch(), "top": BoundedCounter(), "examples": [],
    }
    for name in COL_NAMES
}
amount_profile = NumericProfile()
amount_sample: list[np.ndarray] = []
amount_sample_n = 0
sign_by_direction: Counter = Counter()

date_format: str | None = None
date_parse_fail = 0      # non-null strings that failed to parse
date_min, date_max = None, None
month_counter: Counter = Counter()
date_has_time_component = False

desc_norm_kmv = KMVSketch()
desc_template_counter = BoundedCounter()
desc_template_examples: dict[str, list[str]] = {}

rows_seen = 0
t0 = time.time()
for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE)):
    if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
        break
    rows_seen += batch.num_rows

    for name in COL_NAMES:
        arr = batch.column(name)
        p = profiles[name]
        p["rows"] += len(arr)
        p["nulls"] += arr.null_count

        vc = pc.value_counts(arr.drop_null())
        values = vc.field("values").to_pandas()
        counts = vc.field("counts").to_pandas().to_numpy()
        if len(values) == 0:
            continue

        if pa.types.is_string(arr.type) or pa.types.is_large_string(arr.type):
            empty_mask = values == ""
            if empty_mask.any():
                p["empties"] += int(counts[empty_mask.to_numpy()].sum())

        p["kmv"].update(hash_series(values))
        p["top"].update_pairs(values.tolist(), counts.tolist())
        if len(p["examples"]) < N_EXAMPLES:
            p["examples"].extend(values.head(N_EXAMPLES - len(p["examples"])).tolist())

        # -- near-duplicate template sketch for the description column
        if name == DESC_COL:
            norm = normalise_desc(values)
            grouped = pd.DataFrame({"norm": norm, "raw": values, "n": counts}) \
                        .groupby("norm", sort=False)
            sums = grouped["n"].sum()
            desc_norm_kmv.update(hash_series(sums.index.to_series()))
            desc_template_counter.update_pairs(sums.index.tolist(), sums.tolist())
            if len(desc_template_examples) < TEMPLATE_EXAMPLE_CAP:
                for tmpl, raws in grouped["raw"].unique().items():
                    ex = desc_template_examples.setdefault(tmpl, [])
                    for r in raws[:4]:
                        if r not in ex and len(ex) < 4:
                            ex.append(r)
                    if len(desc_template_examples) >= TEMPLATE_EXAMPLE_CAP:
                        break

    # -- timestamp column
    s_date = batch.column(DATE_COL).to_pandas()
    if date_format is None:
        date_format = detect_date_format(s_date.dropna().head(1000))
        log.info("detected date format: %r", date_format)
    parsed = pd.to_datetime(s_date, format=date_format, errors="coerce")
    date_parse_fail += int((parsed.isna() & s_date.notna()).sum())
    valid = parsed.dropna()
    if len(valid):
        bmin, bmax = valid.min(), valid.max()
        date_min = bmin if date_min is None else min(date_min, bmin)
        date_max = bmax if date_max is None else max(date_max, bmax)
        month_counter.update((valid.dt.year * 100 + valid.dt.month).value_counts().to_dict())
        if not date_has_time_component:
            date_has_time_component = bool(
                (valid.dt.hour.to_numpy() + valid.dt.minute.to_numpy()).any()
            )

    # -- amount column (+ sign x direction cross-tab, stride sample)
    amt = batch.column(AMOUNT_COL).to_pandas().to_numpy(dtype=np.float64)
    amt_valid = amt[~np.isnan(amt)]
    amount_profile.update(amt_valid)
    if amount_sample_n < AMOUNT_SAMPLE_CAP:
        take = amt_valid[::AMOUNT_SAMPLE_STRIDE]
        amount_sample.append(take)
        amount_sample_n += len(take)
    direction = batch.column(DIRECTION_COL).to_pandas().fillna("<null>")
    sign = np.where(np.isnan(amt), "<null>",
                    np.where(amt > 0, "positive", np.where(amt < 0, "negative", "zero")))
    sign_by_direction.update(
        pd.Series(direction.to_numpy() + "|" + sign).value_counts().to_dict()
    )

    if (batch_idx + 1) % 5 == 0 or batch.num_rows < BATCH_SIZE:
        log.info("batch %d done — %s rows (%.0fs elapsed)",
                 batch_idx + 1, f"{rows_seen:,}", time.time() - t0)

amount_sample_arr = (np.concatenate(amount_sample)[:AMOUNT_SAMPLE_CAP]
                     if amount_sample else np.empty(0))
log.info("streaming pass complete: %s rows in %.0fs", f"{rows_seen:,}", time.time() - t0)

## 4. Column profiles — nulls, cardinality, sample values

In [ ]:
rows = []
for name in COL_NAMES:
    p = profiles[name]
    n = p["rows"]
    distinct = p["kmv"].estimate()
    top5 = p["top"].counter.most_common(5)
    rows.append({
        "column": name,
        "arrow_type": str(schema.field(name).type),
        "rows": n,
        "null_count": p["nulls"],
        "null_rate": round(p["nulls"] / n, 4) if n else None,
        "empty_rate": round(p["empties"] / n, 4) if n else None,
        "distinct_approx": distinct,
        "distinct_is_exact": p["kmv"].is_exact,
        "top_values_truncated": p["top"].truncated,
        "top_5_values": "; ".join(f"{v!r}×{c:,}" for v, c in top5)[:160],
        "example_values": "; ".join(repr(v)[:40] for v in p["examples"][:4]),
    })

column_profile_df = pd.DataFrame(rows)

column_profiles = {
    r["column"]: {
        **{k: r[k] for k in ("arrow_type", "rows", "null_count", "null_rate",
                             "empty_rate", "distinct_approx", "distinct_is_exact",
                             "top_values_truncated")},
        "top_20_values": [
            {"value": v, "count": int(c)}
            for v, c in profiles[r["column"]]["top"].counter.most_common(20)
        ],
        "example_values": profiles[r["column"]]["examples"],
    }
    for r in rows
}

with pd.option_context("display.max_colwidth", 80):
    display(column_profile_df)

## 5. Timestamp audit

PRAGMA encodes each event's time twice: log-seconds since the previous event and
intraday/weekly calendar cycles (hour-of-day, day-of-week, day-of-month). This
section checks what temporal resolution the data can actually support.

In [ ]:
month_hist = pd.Series(month_counter).sort_index()
month_hist.index = month_hist.index.map(lambda ym: f"{ym // 100:04d}-{ym % 100:02d}")
expected_months = pd.period_range(str(date_min)[:7], str(date_max)[:7], freq="M") \
    .astype(str) if date_min is not None else []
missing_months = [m for m in expected_months if m not in month_hist.index]

timestamp_report = {
    "column": DATE_COL,
    "detected_format": date_format,
    "granularity": "intraday" if date_has_time_component else "day (no time-of-day)",
    "parse_failures_nonnull": int(date_parse_fail),
    "null_count": profiles[DATE_COL]["nulls"],
    "min_date": str(date_min), "max_date": str(date_max),
    "months_covered": int(len(month_hist)),
    "months_missing_in_range": missing_months,
    "rows_per_month": {m: int(c) for m, c in month_hist.items()},
}
log.info("dates: %s -> %s | format=%s | granularity=%s | parse failures=%d",
         date_min, date_max, date_format, timestamp_report["granularity"], date_parse_fail)
print(json.dumps({k: v for k, v in timestamp_report.items() if k != "rows_per_month"},
                 indent=2, default=str))
try:
    import matplotlib  # noqa: F401
    month_hist.plot(kind="bar", figsize=(14, 3), title="Rows per month")
except ImportError:
    scale = max(month_hist.max() // 60, 1)
    print("\nRows per month:")
    for m, c in month_hist.items():
        print(f"  {m} {'█' * int(c // scale):60s} {c:>12,}")

## 6. Amount audit

Checks magnitude distribution, zero/negative rates, and whether amounts are
signed or rely on the debit/credit flag for direction. Also notes that **no
currency column exists** — PRAGMA tokenises `Currency` explicitly.

In [ ]:
q = ([0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
     if len(amount_sample_arr) else [])
amount_report = {
    "column": AMOUNT_COL,
    **amount_profile.summary(),
    "null_count": profiles[AMOUNT_COL]["nulls"],
    "quantiles_from_stride_sample": {
        str(p): float(np.quantile(amount_sample_arr, p)) for p in q
    },
    "quantile_sample_size": int(len(amount_sample_arr)),
    "currency_column_present": any("currenc" in c.lower() for c in COL_NAMES),
}

cross = (pd.Series(sign_by_direction)
         .rename_axis("direction|sign").sort_values(ascending=False))
cross_df = cross.reset_index()
cross_df[["direction", "amount_sign"]] = cross_df["direction|sign"].str.split("|", expand=True)
sign_direction_table = cross_df.pivot_table(
    index="direction", columns="amount_sign", values=0, aggfunc="sum", fill_value=0
)
amount_report["sign_by_direction"] = {
    k: int(v) for k, v in sign_by_direction.items()
}

print(json.dumps({k: v for k, v in amount_report.items() if k != "sign_by_direction"},
                 indent=2))
sign_direction_table

## 7. Description / merchant / category / subcategory coverage

Includes the known caveat from `docs/transaction_model_strategy.md`: unknown
categories were previously undercounted due to case-sensitive comparison, so all
unknown-label matching here is **case-insensitive**.

In [ ]:
UNKNOWN_LABELS = {
    "miscellaneous/unknown", "unknown", "uncategorized", "uncategorised",
    "classification failed", "misc", "other", "none", "",
}

def label_coverage(col: str) -> dict:
    p = profiles[col]
    n = p["rows"]
    counter = p["top"].counter
    unknown_rows = sum(c for v, c in counter.items()
                       if str(v).strip().lower() in UNKNOWN_LABELS)
    non_null = n - p["nulls"]
    labelled = non_null - unknown_rows - p["empties"]
    return {
        "column": col,
        "rows": n,
        "non_null": non_null,
        "non_null_rate": round(non_null / n, 4),
        "unknown_or_placeholder_rows": int(unknown_rows),
        "usable_labelled_rows": int(max(labelled, 0)),
        "usable_labelled_rate": round(max(labelled, 0) / n, 4),
        "distinct_approx": p["kmv"].estimate(),
        "counts_truncated": p["top"].truncated,
    }

coverage_cols = [DESC_COL, "merchant_name", "Category", "Category_original",
                 "Category_corrected", "subcategory", "matched_rule_name"]
coverage_report = {c: label_coverage(c) for c in coverage_cols if c in COL_NAMES}
coverage_report["matched_rule_id"] = {
    "column": "matched_rule_id",
    "rows": profiles["matched_rule_id"]["rows"],
    "non_null": profiles["matched_rule_id"]["rows"] - profiles["matched_rule_id"]["nulls"],
    "non_null_rate": round(
        1 - profiles["matched_rule_id"]["nulls"] / profiles["matched_rule_id"]["rows"], 4),
}

coverage_df = pd.DataFrame(coverage_report.values())
for c in coverage_cols:
    if c in coverage_report:
        r = coverage_report[c]
        log.info("%s: non-null %.1f%% | usable-labelled %.1f%% | ~%s distinct",
                 c, 100 * r["non_null_rate"],
                 100 * r.get("usable_labelled_rate", r["non_null_rate"]),
                 f"{r.get('distinct_approx', 0):,}")
coverage_df

## 8. Duplicate & near-duplicate transaction descriptions

- **Exact duplicates**: distinct raw descriptions (KMV estimate) vs total rows.
- **Near-duplicates**: descriptions collapsed to templates by normalising
  digits/punctuation (`pos purchase shoprite 12345` → `pos purchase shoprite #`),
  with example clusters where several distinct raw strings share one template.

In [ ]:
desc_p = profiles[DESC_COL]
desc_rows = desc_p["rows"] - desc_p["nulls"]
distinct_raw = desc_p["kmv"].estimate()
distinct_norm = desc_norm_kmv.estimate()

top_exact = [{"description": v, "row_count": int(c)}
             for v, c in desc_p["top"].counter.most_common(30)]
top_templates = [{"template": t, "row_count": int(c),
                  "example_raw": desc_template_examples.get(t, [None])[0]}
                 for t, c in desc_template_counter.counter.most_common(30)]

near_dup_clusters = [
    {"template": t, "row_count": int(c),
     "distinct_raw_examples": desc_template_examples[t]}
    for t, c in desc_template_counter.counter.most_common(2000)
    if len(desc_template_examples.get(t, [])) >= 2
][:20]

duplication_report = {
    "description_rows_non_null": int(desc_rows),
    "distinct_raw_descriptions_approx": int(distinct_raw),
    "distinct_raw_is_exact": desc_p["kmv"].is_exact,
    "exact_duplication_rate": round(1 - distinct_raw / desc_rows, 4) if desc_rows else None,
    "mean_rows_per_distinct_description": round(desc_rows / distinct_raw, 2) if distinct_raw else None,
    "distinct_normalised_templates_approx": int(distinct_norm),
    "near_duplicate_collapse_ratio": round(distinct_raw / distinct_norm, 2) if distinct_norm else None,
    "top_30_exact_descriptions": top_exact,
    "top_30_templates": top_templates,
    "near_duplicate_cluster_examples": near_dup_clusters,
    "note": ("Counts from bounded heavy-hitter counters (approximate for the "
             "long tail); distinct counts are KMV estimates."),
}

log.info("descriptions: %s rows -> ~%s distinct raw (dup rate %.1f%%) -> ~%s templates "
         "(collapse %.1fx)", f"{desc_rows:,}", f"{distinct_raw:,}",
         100 * duplication_report["exact_duplication_rate"],
         f"{distinct_norm:,}", duplication_report["near_duplicate_collapse_ratio"])
print("\nTop exact duplicate descriptions:")
display(pd.DataFrame(top_exact).head(10))
print("Near-duplicate clusters (one template, several raw variants):")
display(pd.DataFrame([
    {"template": c["template"], "rows": c["row_count"],
     "variants": " | ".join(c["distinct_raw_examples"][:3])}
    for c in near_dup_clusters[:10]
]))

## 9. PRAGMA field checklist & stage readiness

PRAGMA needs, per event: timestamp (ideally to the second), typed key–value
fields (`Type`, `Direction`, `Amount`, `Currency`, `Fee`, `Description`, `MCC`),
an **account/customer identifier** to assemble per-customer event histories, and
**profile-state** attributes (plan, region, balance quantile, age band, …).
This cell derives, from the audited columns, which fields exist and which
PRAGMA stages are currently feasible.

In [ ]:
def col_ok(name: str, max_null_rate: float = 0.5) -> bool:
    p = profiles.get(name)
    return bool(p and p["rows"] and (p["nulls"] / p["rows"]) <= max_null_rate)

id_pattern = re.compile(r"(account|customer|user|client|iban|acct|holder).*(id|no|num)?",
                        re.IGNORECASE)
id_candidates = [c for c in COL_NAMES if id_pattern.search(c)]

field_checks = [
    ("event_timestamp", DATE_COL, col_ok(DATE_COL) and date_parse_fail == 0,
     f"present, format {date_format!r}, but "
     + ("intraday resolution available" if date_has_time_component
        else "DAY-level only — no time-of-day for PRAGMA's log-seconds/intraday cycles")),
    ("transaction_amount", AMOUNT_COL, col_ok(AMOUNT_COL),
     f"present; {amount_profile.negatives:,} negative and {amount_profile.zeros:,} zero values; "
     "direction carried by TransactionTypeDebitCredit"),
    ("direction (debit/credit)", DIRECTION_COL, col_ok(DIRECTION_COL), "present"),
    ("currency", None, False, "ABSENT — no currency column; single-currency (NAD) assumed but unverifiable"),
    ("fee", None, False, "ABSENT — no fee column"),
    ("account/customer identifier", ", ".join(id_candidates) or None, bool(id_candidates),
     "ABSENT — no account/customer/user id column; per-customer histories cannot be built"
     if not id_candidates else "candidate columns found — verify semantics"),
    ("event type", "PaymentType/Channel/TransactionParticularCode",
     col_ok("PaymentType") or col_ok("Channel"),
     "partial proxies: PaymentType, Channel/ChannelCode, TransactionParticularCode (high null rates on some)"),
    ("description (free text)", DESC_COL, col_ok(DESC_COL), "present"),
    ("merchant", "merchant_name", col_ok("merchant_name", max_null_rate=1.0),
     f"rule-derived, {100 * (1 - profiles['merchant_name']['nulls'] / profiles['merchant_name']['rows']):.1f}% coverage"),
    ("merchant category code (MCC)", None, False, "ABSENT — no MCC column"),
    ("category / subcategory labels", "Category_corrected, subcategory", col_ok("Category_corrected"),
     "present (rule/model-derived, includes unknown placeholders)"),
    ("product fields", None, False, "ABSENT — no product/account-type column"),
    ("profile state (plan, region, balance, age…)", None, False, "ABSENT — no profile attributes"),
    ("balance", None, False, "ABSENT — no running balance column"),
]
field_df = pd.DataFrame(field_checks, columns=["pragma_field", "mapped_column", "available", "notes"])

has_ts = bool(col_ok(DATE_COL))
has_amount = bool(col_ok(AMOUNT_COL))
has_desc = bool(col_ok(DESC_COL))
has_customer_id = bool(id_candidates)
has_profile_state = False
has_intraday = bool(date_has_time_component)

stage_readiness = {
    "stage0_data_audit": {
        "feasible": True,
        "notes": "This notebook.",
    },
    "stage1_single_event_kv_serialisation": {
        "feasible": has_ts and has_amount and has_desc,
        "notes": ("Key-value-time serialisation of single events is possible with "
                  "date, amount, direction, payment type, channel and description. "
                  "Currency/fee/MCC keys unavailable."),
    },
    "stage2_typed_event_encoder": {
        "feasible": has_ts and has_amount and has_desc,
        "notes": ("A typed event encoder (numeric buckets for amount, categorical "
                  "tokens for direction/payment type/channel, subword text for "
                  "description) can be trained on independent events, but without "
                  "intraday timestamps the calendar-cycle features reduce to "
                  "day-of-week/day-of-month."),
    },
    "stage3_customer_event_histories": {
        "feasible": has_customer_id,
        "notes": ("BLOCKED — no account/customer identifier, so events cannot be "
                  "grouped into per-customer ordered histories."),
    },
    "stage4_profile_state_encoder": {
        "feasible": has_profile_state,
        "notes": "BLOCKED — no profile-state attributes (plan, region, balance, tenure...).",
    },
    "stage5_masked_event_pretraining": {
        "feasible": has_customer_id,
        "notes": ("BLOCKED — masked-event modelling over histories requires stage 3; "
                  "only single-event masking would be possible, which loses PRAGMA's "
                  "cross-event context."),
    },
    "stage6_downstream_probes_lora": {
        "feasible": has_customer_id and has_profile_state,
        "notes": ("BLOCKED — downstream tasks (credit, LTV, recurrence, fraud) are "
                  "defined per customer at an evaluation point; requires stages 3-4."),
    },
}

missing_data_requirements = [
    "account/customer identifier per transaction (hard blocker for event histories)",
    "full timestamps with time-of-day (currently day-level only)",
    "currency code per transaction",
    "merchant category code (MCC)",
    "fee / charge amount per transaction",
    "profile-state attributes: product/plan, region, account tenure, balance or balance quantile, demographic bands",
    "additional event sources (app events, communications) if multi-source histories are wanted",
    "running balance per transaction (optional but valuable)",
]

pragma_readiness = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": str(DATASET_PATH),
    "field_checklist": field_df.to_dict(orient="records"),
    "stage_readiness": stage_readiness,
    "missing_data_requirements": missing_data_requirements,
}

display(field_df)
print("\nStage readiness:")
for stage, info in stage_readiness.items():
    print(f"  {'✅' if info['feasible'] else '⛔'} {stage}: {info['notes']}")

## 10. Persist audit artifacts

Everything is written under `ml_model/data/processed/audit/` only.

In [ ]:
def save_json(obj: dict, path: Path) -> None:
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        return str(o)
    path.write_text(json.dumps(obj, indent=2, default=default))
    log.info("wrote %s (%.1f KB)", path.name, path.stat().st_size / 1024)

summary_metrics = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "dataset": str(DATASET_PATH),
    "rows_profiled": int(rows_seen),
    "rows_in_file": int(meta.num_rows),
    "smoke_test_mode": MAX_BATCHES is not None,
    "columns": len(COL_NAMES),
    "timestamp": {k: v for k, v in timestamp_report.items() if k != "rows_per_month"},
    "amount": {k: v for k, v in amount_report.items() if k != "sign_by_direction"},
    "coverage": {
        c: {k: v for k, v in r.items() if k != "column"}
        for c, r in coverage_report.items()
    },
    "description_duplication": {
        k: v for k, v in duplication_report.items()
        if not k.startswith(("top_", "near_duplicate_cluster"))
    },
}

save_json(schema_report, AUDIT_DIR / "schema_report.json")
save_json(column_profiles, AUDIT_DIR / "column_profiles.json")
save_json({"timestamp": timestamp_report, "amount": amount_report,
           "coverage": coverage_report}, AUDIT_DIR / "field_audits.json")
save_json(duplication_report, AUDIT_DIR / "description_duplication.json")
save_json(pragma_readiness, AUDIT_DIR / "pragma_readiness.json")
save_json(summary_metrics, AUDIT_DIR / "summary_metrics.json")
column_profile_df.to_csv(AUDIT_DIR / "column_profiles.csv", index=False)
pd.DataFrame(top_templates).to_csv(AUDIT_DIR / "near_duplicate_templates.csv", index=False)

report_lines = [
    "# Stage 0 audit report",
    f"- generated: {summary_metrics['generated_at']}",
    f"- dataset: `{DATASET_PATH.name}` ({meta.num_rows:,} rows, "
    f"{DATASET_PATH.stat().st_size / 1e6:.0f} MB, {meta.num_row_groups} row groups)",
    f"- audited via requested primary file: {schema_report['dataset_is_requested_primary']}",
    "",
    "## Key findings",
    f"- Timestamps: {timestamp_report['granularity']}, "
    f"{timestamp_report['min_date']} → {timestamp_report['max_date']}, "
    f"{timestamp_report['parse_failures_nonnull']} parse failures.",
    f"- Amounts: min {amount_report.get('min')}, max {amount_report.get('max'):,.0f}, "
    f"{amount_report.get('negatives', 0):,} negatives; no currency column.",
    f"- Descriptions: {duplication_report['description_rows_non_null']:,} rows → "
    f"~{duplication_report['distinct_raw_descriptions_approx']:,} distinct "
    f"(dup rate {duplication_report['exact_duplication_rate']:.1%}) → "
    f"~{duplication_report['distinct_normalised_templates_approx']:,} templates "
    f"({duplication_report['near_duplicate_collapse_ratio']}x near-dup collapse).",
    f"- merchant_name coverage: {coverage_report['merchant_name']['non_null_rate']:.1%}; "
    f"usable category rate (Category_corrected, excl. unknown placeholders): "
    f"{coverage_report['Category_corrected']['usable_labelled_rate']:.1%}.",
    "",
    "## PRAGMA stage readiness",
]
report_lines += [
    f"- {'FEASIBLE' if info['feasible'] else 'BLOCKED'} — {stage}: {info['notes']}"
    for stage, info in stage_readiness.items()
]
report_lines += ["", "## Missing data requirements"]
report_lines += [f"- {m}" for m in missing_data_requirements]
(AUDIT_DIR / "audit_report.md").write_text("\n".join(report_lines))
log.info("wrote audit_report.md")

print("\nArtifacts in", AUDIT_DIR)
for f in sorted(AUDIT_DIR.iterdir()):
    print(f"  {f.name:40s} {f.stat().st_size / 1024:10.1f} KB")

## 11. Conclusions — what PRAGMA work is possible today

**Possible now with the available fields**

- **Stage 0 (this audit)** and continued single-transaction modelling: the
  dataset supports leakage-aware *key–value serialisation of individual events*
  (date, amount, debit/credit direction, payment type, channel, description) and
  a *typed single-event encoder* with numeric amount bucketing and categorical
  tokens — PRAGMA's tokenisation ideas applied at the single-event level, which
  is exactly the "borrow now" position in `docs/transaction_model_strategy.md`.
- Retrieval/classification heads over frozen single-event embeddings, evaluated
  with the existing benchmark under `data/evaluation/`.

**Blocked until new data arrives**

- **Customer event histories (PRAGMA's core)**: there is *no account/customer
  identifier*, so events cannot be grouped into ordered per-customer sequences —
  the event-history encoder, masked-event pretraining over histories, and every
  per-customer downstream task (credit, LTV, recurrence, fraud) are blocked.
- **Profile-state encoder**: no profile attributes (plan, region, tenure,
  balance quantile, demographics) exist in the dataset.
- **Full temporal encoding**: timestamps are day-level, so PRAGMA's log-seconds
  gaps and hour-of-day cycles cannot be computed; only day-of-week/day-of-month
  calendar features are available.
- Also absent: currency, MCC, fees, product fields, running balances.

**Data acquisition priorities**: (1) pseudonymised account/customer id per
transaction, (2) full timestamps, (3) currency + MCC + fee columns, (4) a
minimal profile-state table keyed by the same id.

See `data/processed/audit/pragma_readiness.json` for the machine-readable
version of this assessment.